# Task 2.1 — Dataset Selection and Setup

**Student:** Yashi Gupta (Roll No. 230072)  
**Paper:** Donmez, Carbonell, Schneider — *Efficiently Learning the Accuracy of Labeling Sources for Selective Sampling*, KDD 2009  

---

## Dataset Justification

We use the **UCI Mushroom** dataset (8,124 samples, 22 categorical features, binary classification: edible vs poisonous) from the OpenML repository. This dataset is appropriate for reproducing IEThresh because (1) the original paper evaluates on UCI binary classification tasks with multiple noisy oracles, and Mushroom is a standard UCI benchmark with a clean binary target; (2) its moderate size allows running many active-learning iterations with a realistic unlabeled pool while keeping computation tractable; and (3) the categorical feature space is well-suited to logistic regression after label encoding, matching the linear classifier setup described in Section 4.1 of the paper. A limitation compared to the original paper is that Donmez et al. use datasets like `mushroom`, `splice`, and `ringnorm` with specific feature preprocessing not fully detailed; our label-encoding of categorical features is a simplification that may slightly alter decision boundaries compared to whatever encoding the authors used.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

SEED = 42
np.random.seed(SEED)
print(f"Random seed set to {SEED}")

Random seed set to 42


In [2]:
# Load Mushroom dataset from OpenML
data = fetch_openml(name='mushroom', version=1, as_frame=True, parser='auto')
X_raw = data.data
y_raw = data.target

# Label-encode all categorical features
# Each column is independently encoded to integer values
X_encoded = X_raw.copy()
for col in X_encoded.columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
X_encoded = X_encoded.values.astype(float)

# Encode target: edible=0, poisonous=1 (or whichever ordering LabelEncoder picks)
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y_raw)

# Save processed data to partB/data/
os.makedirs('data', exist_ok=True)
np.save('data/mushroom_X.npy', X_encoded)
np.save('data/mushroom_y.npy', y_encoded)

print(f"Dataset shape: {X_encoded.shape}")
print(f"Class distribution: {np.bincount(y_encoded)}")
print(f"Classes: {le_target.classes_}")
print(f"Saved to data/mushroom_X.npy and data/mushroom_y.npy")

Dataset shape: (8124, 22)
Class distribution: [4208 3916]
Classes: ['e' 'p']
Saved to data/mushroom_X.npy and data/mushroom_y.npy


## Preprocessing Details

The Mushroom dataset contains 22 categorical features (e.g., cap-shape, odor, gill-size). Since scikit-learn's `LogisticRegression` requires numerical input, we apply **ordinal label encoding** to each feature independently using `sklearn.preprocessing.LabelEncoder`. Each unique category within a feature is mapped to an integer starting from 0.

This is a simplification — one-hot encoding would preserve the nominal nature of the features — but label encoding keeps the feature space compact (22 dimensions vs. potentially 100+), which is closer to the dimensionality the paper likely operated with. The binary target is also label-encoded (the two classes map to 0 and 1).

In [3]:
# 70/30 stratified train/test split (matching paper Section 4.1)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.3, random_state=SEED, stratify=y_encoded
)

# Initial labeled set: 1 positive + 1 negative (Section 4.1)
# The paper states: "we start with one positive and one negative example"
pos_idx = np.where(y_train == 1)[0][0]
neg_idx = np.where(y_train == 0)[0][0]
initial_indices = [pos_idx, neg_idx]
pool_indices = [i for i in range(len(y_train)) if i not in initial_indices]

X_labeled = X_train[initial_indices]
y_labeled = y_train[initial_indices]
X_pool = X_train[pool_indices]
y_pool = y_train[pool_indices]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Initial labeled set: {len(X_labeled)} (1 pos + 1 neg)")
print(f"Unlabeled pool: {len(X_pool)}")
print(f"\nLabeled set class distribution: {np.bincount(y_labeled)}")
print(f"Pool class distribution: {np.bincount(y_pool)}")

Train size: 5686, Test size: 2438
Initial labeled set: 2 (1 pos + 1 neg)
Unlabeled pool: 5684

Labeled set class distribution: [1 1]
Pool class distribution: [2944 2740]


## Matching the Paper's Experimental Protocol (Section 4.1)

Our setup follows the experimental protocol described in Section 4.1 of Donmez et al. (KDD 2009):

- **Train/test split**: 70/30 with stratification to preserve class balance, consistent with standard UCI evaluation protocols.
- **Initial labeled set**: One positive and one negative example, ensuring the classifier can be trained from the first iteration. The paper states: *"we start with one positive and one negative example in the labeled set"*.
- **Unlabeled pool**: The remaining training examples form the pool from which instances are selected via uncertainty sampling.
- **Oracle simulation**: In the next notebook (task 2 2), we simulate `k = 10` oracles with accuracies drawn uniformly from `[0.5, 1.0]`, matching Section 4.1.

The saved `.npy` files in `data/` will be loaded by subsequent notebooks to ensure consistent data across all experiments.